# Stage 01 — Review (T1 + T3 + M1/M2/M3)

For every patent in `matched/` (output of Stage 00b2 — figures already cropped
and matched to BRIEF DESCRIPTION lines):
1. Read figure label from filename (Stage 00b2 encoded it; no re-OCR)
2. Read BRIEF DESCRIPTION from `text/<patent_id>.txt`
3. Match filename label → description line → assign `match_status`
4. SigLIP zero-shot: T2 (per-figure visual fields), G1 (architecture topType),
   M1 (fuselage/gear/symmetry), M2 (wing config/empennage), M3 (propulsion)
5. PatentSBERTa: T1 dimensions (scope, field, target)
6. Write `labels/<patent_id>.json`
7. Generate per-patent review HTML pre-filled with all AI predictions

SigLIP and PatentSBERTa are **required** — the notebook will not run without them.

**match_status colour key:**
- 🟢 `matched` — filename label found and matched to a description line (confidence 0.95)
- 🔴 `unmatched` — filename label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — no label in filename; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")


In [ ]:
import os
from src.config_loader import load_config
cfg = load_config()

# Must run BEFORE any sentence_transformers/open_clip/huggingface_hub import:
# huggingface_hub freezes HF_HUB_CACHE as a constant at first import, so
# setting it afterward is silently ignored and weights re-download to the
# default ~/.cache/huggingface instead of the project models/ folder.
os.environ["HF_HUB_CACHE"] = str(cfg["paths"]["siglip_cache"])
os.environ["HF_HOME"]      = str(cfg["paths"]["siglip_cache"])

from src.matcher import parse_description, match_images, label_from_filename
from src.cross_modal import load_siglip_model, verify_matches

# ── PatentSBERTa — required for T1 dimension classification ──────────────────
# Cached under cfg["paths"]["sbert_cache"] (Patent-Labelling-Tools/models/SBERT)
# so weights stay inside the project instead of ~/.cache/huggingface.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"]))
print("PatentSBERTa ready.")

# ── SigLIP — required for T2/G1/M1/M2/M3 visual classification ───────────────
USE_SIGLIP = True
siglip_bundle = load_siglip_model(cache_dir=cfg["paths"]["siglip_cache"])
print(f"SigLIP ready on {siglip_bundle[3]}.")


In [ ]:
# ── Batch selector ────────────────────────────────────────────────────────
# Set BATCH_ID to the batch you want to review (see data/batches.xlsx Summary sheet).
BATCH_ID = 0

import pandas as pd
from pathlib import Path

cfg = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path = Path(cfg["paths"]["data"]) / "batches.xlsx"

if not batches_path.exists():
    raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

sheet_name  = f"Batch_{BATCH_ID:02d}"
batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")
print(batch_df[["patent_id","company_canonical","prototype_label"]].head(10).to_string(index=False))


In [ ]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
matched_dir = cfg['paths']['matched']
text_dir    = cfg['paths']['text']
labels_dir  = cfg['paths']['labels']
print(f"matched : {matched_dir}")
print(f"text    : {text_dir}")
print(f"labels  : {labels_dir}")


In [ ]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from text/<patent_id>.txt by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

In [ ]:
# ── Single-patent diagnostic (optional) ─────────────────────────────────────
# Set INSPECT_ID to any patent folder name to process just that one patent
# and see its record dict before running the full batch.
# Leave as None to skip.
INSPECT_ID = None  # e.g. "US2015014475A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    data = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    print(f"Figures: {len(data.get('T3_images', []))}")
    print(f"has_splits: {data.get('has_splits')}")


In [ ]:
# ── Batch run configuration ──────────────────────────────────────────────────
LIMIT = None   # int or None — None processes all patents in matched/
# SigLIP and SBERT are always on — see model-loading cell above.
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ── Stage 01 main loop — pre-filtered by crop_quality, confidence-capped for degraded crops ──
# Inlines the same per-patent steps run_stage01() performs (process_patent ->
# build_patent_rows -> export_source_excel) so the crop_quality pre-filter/cap
# logic below can sit directly in the loop, without touching run_stage01()
# itself (it stays available, untouched, for other callers).
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from src import reviewer
from src.excel_schema import build_patent_rows, export_source_excel

# ── Step A — load crops_mapping.csv once (written by 00b2's Cell 5b) ──────────
crops_csv = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
crops_df  = pd.read_csv(crops_csv, dtype=str) if crops_csv.exists() else pd.DataFrame()

# ── Step B — per-patent skip/cap filename sets, keyed by patent_id ────────────
SKIP_QUALITIES = {"blank", "too_small", "rejected", "missing"}
CAP_QUALITIES  = {"low_conf", "merged"}
CAP_VALUE      = 0.55

skip_files_map: dict = {}
cap_files_map:  dict = {}
if not crops_df.empty and "crop_quality" in crops_df.columns:
    for pid, grp in crops_df.groupby("patent_id"):
        skip_files_map[pid] = set(grp.loc[grp["crop_quality"].isin(SKIP_QUALITIES), "output"])
        cap_files_map[pid]  = set(grp.loc[grp["crop_quality"].isin(CAP_QUALITIES),  "output"])

print(f"crop_quality pre-filter: {sum(len(v) for v in skip_files_map.values())} crops to skip, "
      f"{sum(len(v) for v in cap_files_map.values())} crops to confidence-cap "
      f"across {len(skip_files_map)} patents.")


def _cap_confidence(d: dict, cap_value: float = CAP_VALUE) -> None:
    """Recursively clip any 'confidence' > cap_value down to cap_value, in place."""
    if not isinstance(d, dict):
        return
    if isinstance(d.get("confidence"), (int, float)) and d["confidence"] > cap_value:
        d["confidence"] = cap_value
    for v in d.values():
        if isinstance(v, dict):
            _cap_confidence(v, cap_value)


rows = []
all_excel_rows: list[dict] = []
for patent_id in tqdm(patent_ids, desc="Stage 01"):
    try:
        skip = skip_files_map.get(patent_id, set())
        cap  = cap_files_map.get(patent_id, set())

        # ── Step C — skip fully-degraded crops before they ever reach SigLIP/SBERT ──
        record = reviewer.process_patent(
            patent_id, cfg, excel_index, matched_dir,
            sbert_model   = sbert,
            siglip_bundle = siglip_bundle,
            skip_siglip   = not USE_SIGLIP,
            skip_files    = skip,
        )

        # ── Step D — cap SigLIP confidence for degraded-but-kept crops ──────────────
        # Per-figure SigLIP output only exists in T3_images[*]["T2_predictions"]/
        # ["G1_hint"] — M1/M2/M3 predictions are already aggregated to patent level
        # by process_patent() (no per-figure breakdown survives in the returned
        # record), so the cap is applied to T2_predictions/G1_hint only.
        if cap:
            for fig in record.get("T3_images", []):
                if fig.get("file") in cap:
                    _cap_confidence(fig.get("T2_predictions"))
                    _cap_confidence(fig.get("G1_hint"))

        patent_img_dir = reviewer.resolve_patent_image_dir(matched_dir, patent_id)
        all_excel_rows.extend(build_patent_rows(patent_id, record, patent_img_dir))

        figs     = record.get("T3_images", [])
        statuses = [f.get("match_status", "") for f in figs]
        rows.append({
            "patent_id":         patent_id,
            "match_score":       round(
                sum(1 for s in statuses
                    if s in ("matched", "semantic", "positional"))
                / max(len(statuses), 1), 3),
            "matched":           statuses.count("matched"),
            "semantic":          statuses.count("semantic"),
            "positional":        statuses.count("positional"),
            "unmatched":         statuses.count("unmatched"),
            "human_required":    statuses.count("human_required"),
            "has_splits":        record.get("has_splits", False),
            "review_required":   any(f.get("needs_review") for f in figs),
            "description_found": bool(record.get("description_of_drawings")),
            "t2_labeled":        sum(1 for f in figs if f.get("T2_predictions")),
            "total_crops":       len(figs),
            "skipped_crops":     len(skip),
            "capped_crops":      len(cap),
            "error":             None,
        })
    except Exception as exc:
        rows.append({
            "patent_id":   patent_id,
            "error":       str(exc),
            "match_score": 0.0,
            "total_crops": 0,
        })

if all_excel_rows:
    export_source_excel(all_excel_rows, Path(cfg["paths"]["source_excel"]))

summary_df = pd.DataFrame(rows)
print(f"\n{'='*55}")
print(f"  Stage 01 complete: {len(summary_df)} patents")
print(f"  source_patents.xlsx: {len(all_excel_rows)} rows -> {cfg['paths']['source_excel']}")
if "match_score" in summary_df.columns and summary_df["match_score"].notna().any():
    print(f"  Avg match score  : {summary_df['match_score'].mean():.1%}")
    hr = summary_df.get("human_required", pd.Series(dtype=int))
    rr = summary_df.get("review_required", pd.Series(dtype=bool))
    print(f"  Human-required   : {hr.sum() if not hr.empty else 0} crops")
    print(f"  Needs review     : {rr.sum() if not rr.empty else 0} patents")
print(f"{'='*55}")


In [ ]:
print(summary_df.to_string(index=False))

In [ ]:
# ── Single-patent diagnostic — change INSPECT_ID to inspect any patent ─────────
import pandas as pd

INSPECT_ID = None   # e.g. "US2015014475A1"

if INSPECT_ID:
    review_df = pd.read_excel(cfg["paths"]["source_excel"], sheet_name="Review")
    sub = review_df[review_df["Patent_ID"] == INSPECT_ID]
    if sub.empty:
        print(f"No rows found for {INSPECT_ID} in source_patents.xlsx")
    else:
        print(f"\n{INSPECT_ID}  |  {len(sub)} rows")
        print(sub[["Section", "Sub_Dimension", "Field", "Value", "Confidence", "Source"]].to_string(index=False))


In [ ]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Run this after you have finished human review for this batch.
# Only patents with human_review_status == "approved" in batches.xlsx are copied.
# Raw files are never touched. Source is matched/{patent_id}/.

import shutil
from pathlib import Path
import pandas as pd

cfg           = __import__("src.config_loader", fromlist=["load_config"]).load_config()
batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
matched_root  = Path(cfg["paths"]["matched"])
reviewed_root = Path(cfg["paths"]["reviewed"])
sheet_name    = f"Batch_{BATCH_ID:02d}"

batch_df = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
approved = batch_df[batch_df["human_review_status"].str.strip().str.lower() == "approved"]

copied  = 0
skipped = 0

for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()

    src_dir  = matched_root / pid
    dest_dir = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠  matched/{pid}/ not found — skipping")
        skipped += 1
        continue

    dest_dir.mkdir(parents=True, exist_ok=True)

    for f in src_dir.glob("*.png"):
        dest_f = dest_dir / f.name
        if not dest_f.exists():
            shutil.copy2(f, dest_f)
            copied += 1
        else:
            skipped += 1

print(f"Approved patents : {len(approved)}")
print(f"Files copied     : {copied}")
print(f"Already existed  : {skipped}")
print(f"Output root      : {reviewed_root}")


In [ ]:
# ── Collect batch outputs → batch_00/ ─────────────────────────────────────────
# Copies this batch's rows from source_patents.xlsx into batch_00/ for validation.
# When you're happy and run all patents, skip this cell.

import pandas as pd
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / "batch_00"
BATCH_OUT.mkdir(parents=True, exist_ok=True)

review_df = pd.read_excel(cfg["paths"]["source_excel"], sheet_name="Review")
batch_rows = review_df[review_df["Patent_ID"].isin(batch_df["patent_id"])]
out_path = BATCH_OUT / "source_patents.xlsx"
batch_rows.to_excel(out_path, sheet_name="Review", index=False)

print(f"Batch outputs collected → {out_path}")
print(f"  rows : {len(batch_rows)}")


In [ ]:
# ── Collect HTML-wizard JSON exports → reviewed_patents.xlsx ────────────────
# Independent of the cells above — safe to re-run any time. Scans
# html_exports/ for *.json files exported by the master HTML taxonomy wizard
# ("Export JSON & Next Patent" button -> buildExport()), converts each into
# reviewed_patents.xlsx rows via excel_schema, then moves the consumed file
# into html_exports/processed/ so a re-run only picks up new exports.
import json as _json
from pathlib import Path

import pandas as pd

from src.excel_schema import build_patent_rows, append_reviewed_rows
from src.reviewer import resolve_patent_image_dir

# Step 1 - resolve (and persist) the html_review_exports path.
_config_path = repo_root / "config.yaml"
html_exports_dir = cfg["paths"].get("html_review_exports")
if html_exports_dir is None:
    html_exports_dir = Path(cfg["paths"]["data"]) / "html_exports"
    cfg["paths"]["html_review_exports"] = html_exports_dir

    # Append the key to config.yaml's paths: block as a plain text insertion
    # rather than a yaml.safe_load -> yaml.dump round-trip - the latter would
    # silently drop every comment already in config.yaml (it's hand-annotated).
    _raw = _config_path.read_text(encoding="utf-8")
    if "html_review_exports:" not in _raw:
        _lines = _raw.splitlines(keepends=True)
        for _i, _line in enumerate(_lines):
            if _line.strip().startswith("reviewed_excel:"):
                _indent = _line[: len(_line) - len(_line.lstrip())]
                _new_line = _indent + 'html_review_exports: "{{ paths.data }}/html_exports"\n'
                _lines.insert(_i + 1, _new_line)
                break
        _config_path.write_text("".join(_lines), encoding="utf-8")
        print(f"Added paths.html_review_exports -> {html_exports_dir} to {_config_path.name}")

html_exports_dir = Path(html_exports_dir)
processed_dir = html_exports_dir / "processed"
html_exports_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Step 2 - scan for new exports (top-level only - processed/ is a subdirectory
# of html_exports_dir and glob("*.json") is non-recursive, so it's excluded).
json_files = sorted(html_exports_dir.glob("*.json"))

reviewed_xlsx = Path(cfg["paths"]["reviewed_excel"])
matched_dir_for_html = Path(cfg["paths"]["matched"])
ingested = 0


def _hv(value):
    """Wrap a human-entered scalar as the {value, confidence, source} shape
    excel_schema.build_patent_rows() expects (same shape process_patent()'s
    SigLIP/SBERT predictions use), tagged source="human" since it came
    straight from a human-filled wizard export rather than a model."""
    return {"value": value, "confidence": 1.0, "source": "human"} if value is not None else None


def _wizard_to_record(wiz: dict, patent_img_dir: Path) -> dict:
    """
    Translate one buildExport() JSON payload (flat wizard state, keyed by
    figure number, e.g. {"topologyClass":..,"wingConf":..,"figures":{"1":{...}}})
    into the nested record shape build_patent_rows() expects (the same shape
    process_patent() returns: T1/T3_images/G1_prediction/M1_predictions/...).

    Known gap: build_patent_rows() has no field at all for per-wing detail
    (wTilt/wPosV/wPosL/wPlan/wRole) or per-propulsion-card values — its M2
    wing rows and M3 rows are hardcoded blank/single-valued by design (see
    _WING_MANUAL and the single chord/orient/bmech/rmech tuple applied to
    every M3 component in excel_schema.py). The wizard's richer per-wing and
    per-component data is therefore NOT carried into reviewed_patents.xlsx —
    fixing that would mean changing excel_schema.py's schema, out of scope
    here. T1/G1/M1/M2-aggregate/T2-per-figure fields all translate cleanly.
    """
    t1 = {
        "assignee": wiz.get("assignee"), "pub_year": wiz.get("pubYear"), "app_year": wiz.get("appYear"),
    }
    for key, wiz_key in [("scope", "scope"), ("t1Field", "techField"), ("t1Target", "innovTarget")]:
        t1[key] = _hv(wiz.get(wiz_key))

    t3_images = []
    for fig_num, fig in (wiz.get("figures") or {}).items():
        matches = sorted(patent_img_dir.glob(f"*_F{fig_num}.png")) if patent_img_dir.exists() else []
        fname = matches[0].name if matches else None
        t2_predictions = {
            field: _hv(fig.get(field))
            for field in ("per", "acSty", "acCol", "bgSty", "bgCol", "sym")
            if fig.get(field) is not None
        }
        t3_images.append({
            "file": fname, "ocr_label": fig_num, "fig_number": fig_num,
            "matched_description": None, "match_status": "human",
            "match_confidence": 1.0, "needs_review": False,
            "T2_predictions": t2_predictions, "G1_hint": None,
        })

    m1_predictions = {
        "fusShape": _hv(wiz.get("fusShape")), "fusKin": _hv(wiz.get("fusKin")),
        "gearArch": _hv(wiz.get("gearArch")), "latSym": _hv(wiz.get("latSym")),
    }
    m2_predictions = {
        "wingConf": _hv(wiz.get("wingConf")), "empType": _hv(wiz.get("empType")),
        "empKin": _hv(wiz.get("empKin")), "wCount": _hv(wiz.get("wCount")),
    }
    cards = wiz.get("propulsionCards") or []
    first_card = cards[0] if cards else {}
    m3_predictions = {
        "chord": _hv(first_card.get("chord")), "orient": _hv(first_card.get("orient")),
        "bmech": _hv(first_card.get("bmech")), "rmech": _hv(first_card.get("rmech")),
    }

    return {
        "T1": t1,
        "description_of_drawings": None,
        "T3_images": t3_images,
        "has_splits": False,
        "G1_prediction": _hv(wiz.get("topologyClass")),
        "M1_predictions": m1_predictions,
        "M2_predictions": m2_predictions,
        "M3_predictions": m3_predictions,
    }


for jf in json_files:
    wiz_record = _json.loads(jf.read_text(encoding="utf-8"))
    # "recordId" is checked too: the live wizard's buildExport() emits that key
    # (T1_META.recordNumber), not "patentId"/"patent_id" - see
    # UI_for_taxonomy_caracterization_10.0.html's buildExport().
    patent_id = wiz_record.get("patentId") or wiz_record.get("patent_id") or wiz_record.get("recordId")
    if not patent_id:
        print(f"  ⚠ {jf.name}: no patentId/patent_id/recordId key found - skipping")
        continue

    patent_img_dir = resolve_patent_image_dir(matched_dir_for_html, str(patent_id))
    record = _wizard_to_record(wiz_record, patent_img_dir)
    rows = build_patent_rows(str(patent_id), record, patent_img_dir)
    append_reviewed_rows(rows, reviewed_xlsx)

    jf.rename(processed_dir / jf.name)
    ingested += 1
    print(f"  ✓ Ingested {patent_id} from {jf.name}")

if reviewed_xlsx.exists():
    _reviewed_df = pd.read_excel(reviewed_xlsx, sheet_name="Review")
    _n_patents = _reviewed_df["Patent_ID"].nunique()
else:
    _n_patents = 0

print(f"Ingested {ingested} JSON exports -> reviewed_patents.xlsx now contains {_n_patents} patents.")
